# App 7 · LLMOps — 让钱、延迟、错误都看得见

凌晨 3 点收到告警：过去一小时 token 消耗暴涨 40 倍。打开后台一看，QPS 没异常、用户没投诉。问题在哪？

如果你的 LLM 系统没有 trace，这个问题没法答。token 消耗是个标量，告诉你"花了多少钱"但不告诉你"花在哪一步"。可能是某个 prompt 模板被人改坏了，把 `{context}` 写成了 `{{context}}`，导致每次都把整个文档库塞进 prompt；可能是 retry 逻辑死循环；可能是某个 RAG 配置 chunk_size 写错了。这些根因，没有 span tree 就只能靠猜。

LLM 系统的可观测性比普通 web 后端难，难在四点：**每次调用花钱**（不只是延迟）、**延迟天然慢**（看 p95/p99 而不是 avg）、**失败模式多**（hallucination、限流、超时、bad output）、**质量难量化**（HTTP 200 不代表答对）。这一节我们用 `utils/observability.py` 的 `@observe` 装饰器和 `with span()` 上下文管理器把这四个维度接到 trace 树里——同样的 API 在没装 Langfuse 时走 in-memory 的 `MockObserver`，装了 Langfuse 时自动切到真后端。

> **跑这一节前**：跑过 App5 / App6（理解 MCP 工具调用和 Skill 路由——这些都是值得追踪的"能产生 span 的事件"）。可选 SDK `pip install 'langfuse>=2.0'` 装上能看到真 dashboard，不装就走 mock。

In [1]:
# 自动定位 repo 根目录，让 utils 可以 import
import os, sys
_cur = os.path.abspath("")
_root = None
for _c in [_cur, os.path.dirname(_cur), os.path.dirname(os.path.dirname(_cur))]:
    if os.path.isdir(os.path.join(_c, "utils")) and os.path.isfile(os.path.join(_c, "README.md")):
        _root = _c; break
if _root is None:
    raise RuntimeError("找不到 repo 根目录")
os.chdir(_root); sys.path.insert(0, _root)
print(f"[DIR] repo root: {_root}")


[DIR] repo root: C:\Users\lvbab\Documents\GitHub\LLM-Agent-Core_Concept_Code


In [2]:
# 导入：所有 5 天组件 + 新加 observability
from utils.config import setup
env = setup()
from utils.multi_agent import Message, MessageType, BaseAgent, Orchestrator
from utils.mcp_helpers import EduMCPServer, EduMCPClient, ToolDef, tool_from_function
from utils.embedding_backend import SimpleVectorStore
from utils.observability import observer, observe, span, get_backend
import json, time, sys
from pathlib import Path

# 引入 Day5 上午写的 Agentic RAG 工具 (重新建 vector store + bm25)
sys.path.insert(0, str(Path('Applications/mcp_server_demo')))

llm = env.get_llm()
embedder = env.get_embedder()

print(f"OK 5 天组件全就位")
print(f"OK Observability backend: {get_backend()}  (mock=本地; langfuse=已配 LANGFUSE_*)")


[OK] 已加载配置: C:\Users\lvbab\Documents\GitHub\LLM-Agent-Core_Concept_Code\.env
课程环境配置:
  API Key:   ✓ 已配置
  LLM:       dashscope / qwen-plus-2025-01-25
  Embedding: dashscope / text-embedding-v3


[LLM] dashscope / qwen-plus-2025-01-25


[Embedding] dashscope / text-embedding-v3 (dim=1024)
OK 5 天组件全就位
OK Observability backend: mock  (mock=本地; langfuse=已配 LANGFUSE_*)


---

## LLMOps 可观测性

### 3 大支柱

| 支柱 | 能回答的问题 | 对应工具 |
|---|---|---|
| **Trace（调用链）** | 这条用户请求经过了哪些 LLM/tool 调用？哪步慢？ | Langfuse / LangSmith |
| **Metric（指标）** | 整体 QPS / token 消耗 / p95 延迟？ | Prometheus / Datadog |
| **Alert（告警）** | 失败率突变？token 暴涨？ | PagerDuty / 企业微信 |

LLM 调用 vs 普通 web 后端的特殊性：
- **每次调用花钱**：没 trace 你不知道谁烧的
- **延迟天然慢**：1-10s 是常态；要看 p95/p99 而不是 avg
- **失败模式多**：超时 / 限流 / 模型生成 bad output / tool 失败 / hallucination
- **质量难量化**：HTTP 200 不代表答对；要 LLM-as-Judge / 人工抽检

### 我们今天用什么

`utils/observability.py` 已写好两个后端：
- **Langfuse**（如果已 `pip install langfuse` + 配 env）→ 真实 dashboard
- **MockObserver**（默认）→ 内存记录，本地 print 看


In [3]:
# 演示：用 @observe 自动追踪
observer.reset()

@observe("retrieve_doc")
def my_retrieve(query):
    time.sleep(0.05)  # 模拟检索延迟
    return f"docs for '{query}'"

@observe("generate_answer")
def my_generate(query, docs):
    time.sleep(0.1)
    return f"Answer based on {docs[:30]}"

@observe("rag_pipeline")
def my_rag(query):
    docs = my_retrieve(query)
    return my_generate(query, docs)

# 跑一次
result = my_rag("什么是 LoRA？")
print(f"Result: {result}\n")

print("Trace tree:")
observer.print_tree()
print("\nSummary:")
print(json.dumps(observer.summary(), indent=2))


Result: Answer based on docs for '什么是 LoRA？'

Trace tree:
├─ rag_pipeline (151ms)
  ├─ retrieve_doc (51ms)
  ├─ generate_answer (100ms)

Summary:
{
  "n_traces": 1,
  "n_total_spans": 3,
  "total_duration_ms": 150.8,
  "tokens": {
    "prompt": 15,
    "completion": 30,
    "total": 45
  },
  "cost_usd": 0.0
}


In [4]:
# [LEARNER_FILL] 难度=基础+进阶 | 提示=约1-14行 | 参考实现见 enterprise_5days/student/Day5_下午_LLMOps与生产Capstone.ipynb
# ============================================================
# 练习 1 | @observe 包装 LLM 调用 + 计时
# ============================================================
#
# 【基础】（人人必做，10 min）
#   实现 traced_llm_call(prompt)：用 @observe("llm.generate") 包装一次 LLM 调用
#   返回 LLM 的输出
#
# 【进阶】（技术学员选做，10 min）
#   实现 traced_pipeline(query)：组合 retrieve → generate，每步独立 trace
#   - retrieve 用 vector_store（如有）或 mock
#   - 用 with span() 手动控制；记录 token 数 (估算 = len(prompt) / 4)
# ============================================================

@observe("llm.generate")
def traced_llm_call(prompt):
    """【基础】wrap LLM call with trace"""
    # ↓↓↓ 【基础】填空（约 1 行）↓↓↓
    return llm.generate(prompt, temperature=0.1).strip()
    # ↑↑↑ 【基础】结束 ↑↑↑


def traced_pipeline(query):
    """【进阶】retrieve + generate 各自 span，记录 token 估算"""
    # ↓↓↓ 【进阶】填空（约 14 行）↓↓↓
    with span("pipeline.full", input={"query": query}) as p:
        # Retrieve step
        with span("pipeline.retrieve", input={"query": query}) as r:
            time.sleep(0.05)  # mock
            docs = f"mock docs about '{query}'"
            observer.record_tokens(prompt_tokens=len(query) // 4, completion_tokens=0)
        # Generate step
        prompt = f"基于 {docs} 答 {query}"
        with span("pipeline.generate", input={"prompt_len": len(prompt)}) as g:
            answer = llm.generate(prompt, temperature=0.1).strip()
            observer.record_tokens(
                prompt_tokens=len(prompt) // 4,
                completion_tokens=len(answer) // 4,
                cost_usd=0.0001,  # mock pricing
            )
        return answer
    # ↑↑↑ 【进阶】结束 ↑↑↑


def verify():
    print("=" * 56); print("【基础】traced_llm_call"); print("=" * 56)
    try:
        observer.reset()
        ans = traced_llm_call("什么是 RAG？一句话")
        assert isinstance(ans, str) and len(ans) > 0
        print(f"  LLM 答: {ans[:120]}")
        print(f"  Trace 数: {len(observer.spans)}")
        observer.print_tree()
        assert len(observer.spans) >= 1
        print("OK 基础通过\n")
    except NotImplementedError:
        print("SKIP 基础未实现\n"); return
    except Exception as e:
        print(f"FAIL 基础未通过: {type(e).__name__}: {e}\n"); return

    print("=" * 56); print("【进阶】traced_pipeline (含 span + tokens)"); print("=" * 56)
    try:
        observer.reset()
        ans = traced_pipeline("LoRA 与全参微调差别？")
        print(f"  Answer: {ans[:120]}")
        print(f"\n  Trace tree:")
        observer.print_tree()
        summary = observer.summary()
        print(f"\n  Summary: {json.dumps(summary, indent=2)}")
        assert summary["tokens"]["total"] > 0
        assert summary["n_total_spans"] >= 3  # full + retrieve + generate
        print("OK 进阶通过")
    except NotImplementedError:
        print("SKIP 进阶跳过（未实现）")
    except Exception as e:
        print(f"FAIL 进阶未通过: {type(e).__name__}: {e}")

verify()


【基础】traced_llm_call


  LLM 答: RAG（Retrieval-Augmented Generation）是一种结合了信息检索和生成式模型的技术，能够从外部数据库或文档中检索相关信息并生成准确、上下文相关的回答。
  Trace 数: 1
├─ llm.generate (1530ms)
OK 基础通过

【进阶】traced_pipeline (含 span + tokens)


  Answer: LoRA（Low-Rank Adaptation）与全参微调（Full Fine-Tuning）是两种不同的模型微调方法，它们在参数更新、计算资源需求和应用场景等方面存在显著差异。以下是两者的详细对比：

---

### 1. **核心思

  Trace tree:
├─ pipeline.full (33942ms)
  ├─ pipeline.retrieve (51ms)
  ├─ pipeline.generate (33891ms)

  Summary: {
  "n_traces": 1,
  "n_total_spans": 3,
  "total_duration_ms": 33941.7,
  "tokens": {
    "prompt": 15,
    "completion": 479,
    "total": 494
  },
  "cost_usd": 0.0001
}
OK 进阶通过


In [5]:
# [LEARNER_FILL] 难度=基础+进阶 | 提示=约3-6行 | 参考实现见 enterprise_5days/student/Day5_下午_LLMOps与生产Capstone.ipynb
# ============================================================
# 练习 2 | Multi-Agent 间的 trace_id 传递
# ============================================================
#
# 【基础】（人人必做，10 min）
#   定义 2 个 Agent (PlannerAgent, WorkerAgent)，每个都用 @observe 包装其 receive()
#   PlannerAgent → WorkerAgent 顺序调用
#
# 【进阶】（技术学员选做，10 min）
#   实现 trace tree 可视化：
#   - 用 with span("user_request") as root: 包裹整个流程
#   - 让 Planner 和 Worker 的 span 都嵌套在 root 下
#   - 最后输出 tree 看到嵌套关系
# ============================================================

@observe("planner.run")
def planner_run(task):
    """【基础】Planner 接收任务"""
    return llm.generate(f"把『{task}』拆成 2 个子任务，编号列出。", temperature=0.1).strip()


@observe("worker.run")
def worker_run(plan):
    """【基础】Worker 执行计划"""
    return llm.generate(f"按计划执行：\n{plan}\n\n给出简短结果。", temperature=0.2).strip()


def basic_two_agent_pipeline(task):
    """【基础】依次调 planner → worker"""
    # ↓↓↓ 【基础】填空（约 3 行）↓↓↓
    plan = planner_run(task)
    result = worker_run(plan)
    return {"plan": plan, "result": result}
    # ↑↑↑ 【基础】结束 ↑↑↑


def two_agent_with_root_span(task):
    """【进阶】整体放入 root span，看到嵌套 trace tree"""
    # ↓↓↓ 【进阶】填空（约 6 行）↓↓↓
    with span("user_request", input={"task": task}) as root:
        plan = planner_run(task)
        result = worker_run(plan)
        root.update(output={"plan_len": len(plan), "result_len": len(result)})
        return {"plan": plan, "result": result, "root_span": "user_request"}
    # ↑↑↑ 【进阶】结束 ↑↑↑


def verify():
    print("=" * 56); print("【基础】basic_two_agent_pipeline"); print("=" * 56)
    try:
        observer.reset()
        r = basic_two_agent_pipeline("写一个简短的 Python 缓存装饰器")
        print(f"  Plan: {r['plan'][:100]}")
        print(f"  Result: {r['result'][:100]}")
        print(f"\n  Trace tree:")
        observer.print_tree()
        # 应有 2 个根级 span (planner + worker)
        assert len(observer.spans) == 2
        print("OK 基础通过\n")
    except NotImplementedError:
        print("SKIP 基础未实现\n"); return
    except Exception as e:
        print(f"FAIL 基础未通过: {type(e).__name__}: {e}\n"); return

    print("=" * 56); print("【进阶】嵌套 root span"); print("=" * 56)
    try:
        observer.reset()
        r = two_agent_with_root_span("写一个简短的 Python 缓存装饰器")
        print(f"  Trace tree (含 root span 嵌套):")
        observer.print_tree()
        # 应只有 1 个根 span，下面套 2 个 children
        assert len(observer.spans) == 1
        root = observer.spans[0]
        assert len(root.children) == 2
        print(f"\n  根 span '{root.name}' 含 {len(root.children)} 个子调用")
        print("  提示: 生产场景下，trace_id 会跨进程透传 (用 OTEL header 传)，这样微服务也能看完整 trace")
        print("OK 进阶通过")
    except NotImplementedError:
        print("SKIP 进阶跳过（未实现）")
    except Exception as e:
        print(f"FAIL 进阶未通过: {type(e).__name__}: {e}")

verify()


【基础】basic_two_agent_pipeline


  Plan: 1. **设计缓存逻辑**：确定如何存储缓存数据（例如使用字典），并定义缓存键值对的生成规则（通常基于函数参数）。
2. **实现装饰器函数**：创建一个装饰器，将缓存逻辑应用到目标函数上，确保在函数
  Result: 以下是一个简单的缓存逻辑设计及装饰器实现：

```python
from functools import wraps

# 1. 设计缓存逻辑：使用字典存储缓存数据
cache = {}

# 2

  Trace tree:
├─ planner.run (3371ms)
├─ worker.run (9148ms)
OK 基础通过

【进阶】嵌套 root span


  Trace tree (含 root span 嵌套):
├─ user_request (16078ms)
  ├─ planner.run (3980ms)
  ├─ worker.run (12099ms)

  根 span 'user_request' 含 2 个子调用
  提示: 生产场景下，trace_id 会跨进程透传 (用 OTEL header 传)，这样微服务也能看完整 trace
OK 进阶通过


## 4. Token 估算 + 成本可见

LLM 系统比 web 后端"贵"得多——一次完整 RAG 调用动辄几千 token，按 GPT-4 价格算就是几美分。如果 token 用量看不见，**你不知道一个 query 烧了多少钱、哪个 prompt 模板成本高、哪条链路有优化空间**。

`@observe` 自动累计每次调用的 input/output token 估算 + 按 mock pricing 算成本。在 `MockObserver.summary()` 里你能看到一个简单的汇总——n_traces / n_total_spans / 总耗时 / token / cost_usd。生产系统会把这套数据接到 dashboard，做 per-user / per-tenant / per-feature 的成本归因。

下面跑一次看。

## 5. 收尾：从 Mock 到 Langfuse

如果你想把这套接到真 Langfuse dashboard：

```bash
# 1. 注册 https://langfuse.com 或本地起 docker
docker compose up langfuse

# 2. 设置 env var
export LANGFUSE_PUBLIC_KEY=pk_...
export LANGFUSE_SECRET_KEY=sk_...
export LANGFUSE_HOST=http://localhost:3000

# 3. 重跑——utils/observability.py 自动检测，@observe 走真后端
```

之后 mock 的 trace 树会上报到 Langfuse dashboard，能看到：每条对话的完整 trace + token + 延迟、Top-N 慢路径、失败率突变告警。

---

走完这一节你应该明白三件事：

**第一，trace 是源头**——metric 和 alert 都是 trace 数据的聚合派生。先把每次 LLM 调用都包进 span 树，后面想加 metric 加 alert 都很轻。

**第二，装饰器 + 上下文管理器是接入 trace 的最干净 API**。`@observe` 和 `with span()` 的组合覆盖了 90% 的 trace 需求，业务代码几乎不用改。

**第三，token 看不见 = 钱看不见**。LLM 系统的成本分析是 web 后端没有的新维度，必须从一开始就埋进 trace。

下一节 [App8 Capstone](./App8_Production_Capstone.ipynb) 把前面 7 节的全部能力（Multi-Agent / MCP / RAG / Skills / LLMOps）合成一个 production pipeline，跑 batch eval 看综合表现。

In [6]:
# 自检：trace tree / token 估算 / 嵌套 span 是否都体验过了
def verify_app7() -> bool:
    print("=" * 56)
    print("自检 · App7 LLMOps 可观测性")
    print("=" * 56)
    checks: list[tuple[str, bool, str]] = []

    try:
        summary = observer.summary()  # noqa: F821 —— utils.observability 全局单例
        checks.append(("observer 收到 ≥1 个 trace", summary["n_traces"] >= 1,
                       f"n_traces={summary['n_traces']}"))
        checks.append(("trace 包含 ≥3 个 span（嵌套结构）",
                       summary["n_total_spans"] >= 3,
                       f"n_total_spans={summary['n_total_spans']}"))
        checks.append(("token 估算非零（@observe 自动累计）",
                       summary["tokens"]["total"] > 0,
                       f"total={summary['tokens']['total']}"))
        has_nested = any(len(root.children) > 0 for root in observer.spans)  # noqa: F821
        checks.append(("trace tree 有真嵌套（root → child）",
                       has_nested,
                       "嵌套生效" if has_nested else "全是平级"))
    except (NameError, AttributeError, KeyError):
        checks.append(("observer 可用", False, "SKIP §2 / §3 cell 未跑或类型异常"))

    passed = sum(1 for _, ok, _ in checks if ok)
    for name, ok, detail in checks:
        icon = "OK" if ok else ("SKIP" if detail.startswith("SKIP") else "FAIL")
        print(f"  {icon} {name}  ({detail})")
    print(f"\n通过 {passed}/{len(checks)}")
    if passed == len(checks):
        print("下一节：App8_Production_Capstone")
    elif passed >= 2:
        print("部分通过——把跳过的 cell 跑完后重跑这一格。")
    else:
        print("未通过——回到顶部按顺序 Run All。")
    return passed == len(checks)


verify_app7()


自检 · App7 LLMOps 可观测性
  OK observer 收到 ≥1 个 trace  (n_traces=1)
  OK trace 包含 ≥3 个 span（嵌套结构）  (n_total_spans=3)
  OK token 估算非零（@observe 自动累计）  (total=336)
  OK trace tree 有真嵌套（root → child）  (嵌套生效)

通过 4/4
下一节：App8_Production_Capstone


True